In [1]:
import pandas as pd
import numpy as np
import openai
import pickle
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from dotenv import load_dotenv
load_dotenv()
openai.api_key = os.getenv("OPENAI_KEY")

In [2]:
np.random.seed(42)
random_state=42

## Setting Up Functions

In [3]:
df_info = pd.read_csv('dataset_info.csv')
df_info.head(2)

,Dataset Name,Note,Study Title,Type,Topic,# Abstracts,# Included Abstracts,Inclusion Criteria,Exclusion Criteria,DOI
0,ivm.csv,NaN,Efficacy and safety of ivermectin for the trea...,Systematic Review and Meta-Analysis,"COVID-19 Treatment, Antiviral, Pharmacological...",314,35,- Randomized controlled trials and non-randomi...,- Studies that used ivermectin for prophylaxis...,10.1093/qjmed/hcab247
1,ssri.csv,Number may not match the publication due to un...,Efficacy and safety of selective serotonin reu...,Systematic Review and Meta-Analysis,"COVID-19 Treatment, Pharmacological Study","4,019",29,- Randomized controlled trials and non-randomi...,- Studies assessing patients with past or ongo...,10.1016/j.cmi.2023.01.010


In [4]:
def return_criteria(df, instructions, inclusion_criteria, exclusion_criteria, show_prompt=False):
    title = df['title']
    abstract = df['abstract']
    prompt = f'{instructions}\n\nTitle: {title}\n\nAbstract:\n{abstract}\n\nInclusion criteria:\n{inclusion_criteria}\n\nExclusion criteria:\n{exclusion_criteria}\n\nDecision:',
    if show_prompt:
        print(prompt[0])

    memory = [{"role": "user", "content": prompt[0]}]
    try:
        response = openai.ChatCompletion.create(
            model='gpt-3.5-turbo',
            messages=memory,
            temperature=0,
            top_p=0.5,
        )
    except:
        print('Context length exceeded, attempting with GPT-4.')
        response = openai.ChatCompletion.create(
            model='gpt-4',
            messages=memory,
            temperature=0,
            top_p=0.5,
        )
        print('Successfully completed with GPT-4.')
    return response['choices'][0]['message']['content'].strip('.').lower().capitalize()

In [5]:
def loop_through_df(df, instructions, save_ckpt=True, sample=False, sample_n=3, random_state=42):
    dfs = dict()
    for ind, row in df.iterrows():
        print(f'Running: {row["Dataset Name"]}')
        if sample:
            temp_df = pd.read_csv(f"data/{row['Dataset Name']}").sample(sample_n, random_state=random_state)
        else:
            temp_df = pd.read_csv(f"data/{row['Dataset Name']}")
        inclusion_criteria = row['Inclusion Criteria']
        exclusion_criteria = row['Exclusion Criteria']
        try:
            temp_df['decision'] = temp_df.apply(lambda x: return_criteria(
                x, instructions, inclusion_criteria, exclusion_criteria), axis=1)
        except:
            print(f'Error processing {row["Dataset Name"]}')
            continue
        temp_df['decision'] = temp_df['decision'].apply(
            lambda x: "Included" if "Included" in x else "Excluded" if "Excluded" in x else "Undecided")
        dfs[row['Dataset Name']] = temp_df
        # save checkpoint
        if save_ckpt:
            temp_df.to_csv(f'results/{row["Dataset Name"]}')
            with open(f'checkpoints/{row["Dataset Name"]}.pkl', 'wb') as f:
                pickle.dump(temp_df, f)
    return dfs

## [CAUTION: CALLING API] Running Sampled Analysis

In [6]:
instructions = \
"""Instructions:
You are a researcher rigorously screening titles and abstracts of scientific papers for inclusion or exclusion in a review paper. Use the criteria below to inform your decision. If any exclusion criteria are met or not all inclusion criteria are met, exclude the article. If all inclusion criteria are met, include the article. Only type "included" or "excluded" to indicate your decision. Do not type anything else."""

In [23]:
# sampled_dfs = loop_through_df(df_info, instructions, save_ckpt=False, sample=True, sample_n=2)

Running: noa.csv


In [24]:
# for k, v in sampled_dfs.items():
#     print(k)
#     print(v)
#     print('='*20)

noa.csv
                                                  title  \
4860  Small-dose dexamethasone improves patient-perc...   
8639  Does dutasteride make MRI visible lesions less...   

                                               abstract       tag  decision  
4860  Introduction : Previous studies of steroid use...  Included  Included  
8639                                                NaN  Excluded  Excluded  


## Running On Entire Dataset

In [25]:
# CAUTION: Expensive and time-consuming call to the GPT API
# df_final = loop_through_df(df_info, instructions, save_ckpt=True, sample=False)

Running: noa.csv
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded, attempting with GPT-4.
Context length exceeded

In [26]:
# for key, value in df_final.items():
#     value.to_csv(f'results/noa_result.csv', index=False)

## Analysis

### Helper Functions

In [7]:
import pandas as pd

In [8]:
def get_absolute_accuracy(df):
    return df[df['tag'] == df['decision']].shape[0] / df.shape[0]

In [9]:
from sklearn.metrics import classification_report
def get_classification_report(df):
    return classification_report(df['tag'], df['decision'])

In [10]:
from sklearn.metrics import recall_score
def get_recall_score(df, tag):
    return recall_score(df['tag'], df['decision'], pos_label=tag)

In [11]:
def replace_undecided(df):
    df['decision'] = df['decision'].apply(lambda x: 'Excluded' if x == 'Undecided' else x)
    df['decision'] = df['decision'].apply(lambda x: x.capitalize())
    df['tag'] = df['tag'].apply(lambda x: x.capitalize())
    df['decision'] = df['decision'].apply(lambda x: "Included" if "Included" in x else "Excluded" if "Excluded" in x else "Undecided")
    return df

### Analyzing

In [12]:
df_ivm = replace_undecided(pd.read_csv('results/ivm_result.csv'))
df_lpvr = replace_undecided(pd.read_csv('results/lpvr_result.csv'))
df_ssri = replace_undecided(pd.read_csv('results/ssri_result.csv'))
df_raynaud = replace_undecided(pd.read_csv('results/raynaud_result.csv'))
df_noa = replace_undecided(pd.read_csv('results/noa_result.csv'))
df_llm = replace_undecided(pd.read_csv('yejean_results/df_with_decisions.csv'))

In [13]:
print(get_absolute_accuracy(df_ivm))
print(get_absolute_accuracy(df_lpvr))
print(get_absolute_accuracy(df_ssri))
print(get_absolute_accuracy(df_raynaud))
print(get_absolute_accuracy(df_noa))
print(get_absolute_accuracy(df_llm))

0.7484076433121019
0.8461538461538461
0.9492284718765555
0.9651898734177216
0.8952727272727272
0.9429657794676806


In [14]:
# # very verbose
# print(get_classification_report(df_ivm))
# print(get_classification_report(df_lpvr))
# print(get_classification_report(df_ssri))
# print(get_classification_report(df_llm))

In [15]:
print(get_recall_score(df_ivm, 'Included'))
print(get_recall_score(df_lpvr, 'Included'))
print(get_recall_score(df_ssri, 'Included'))
print(get_recall_score(df_raynaud, 'Included'))
print(get_recall_score(df_noa, 'Included'))
print(get_recall_score(df_llm, 'Included'))

0.6857142857142857
0.5934065934065934
0.9655172413793104
0.8333333333333334
0.7824858757062146
1.0


In [16]:
print(get_recall_score(df_ivm, 'Excluded'))
print(get_recall_score(df_lpvr, 'Excluded'))
print(get_recall_score(df_ssri, 'Excluded'))
print(get_recall_score(df_raynaud, 'Excluded'))
print(get_recall_score(df_noa, 'Excluded'))
print(get_recall_score(df_llm, 'Excluded'))

0.7562724014336918
0.8619505494505495
0.9491100526447731
0.9660297239915074
0.8979757633200189
0.9425087108013938


In [17]:
# combine all dfs
df_all = pd.concat([df_ivm, df_lpvr, df_ssri, df_raynaud, df_noa, df_llm], ignore_index=True)

print('='*55)
print(f"Absolute accuracy: {get_absolute_accuracy(df_all)}")
print(f"Recall for Included: {get_recall_score(df_all, 'Included')}")
print(f"Recall for Excluded: {get_recall_score(df_all, 'Excluded')}")
print('='*55)
print(get_classification_report(df_all))
print('='*55)

Absolute accuracy: 0.9073052928154558
Recall for Included: 0.7639405204460966
Recall for Excluded: 0.9104784629941992
              precision    recall  f1-score   support

    Excluded       0.99      0.91      0.95     24307
    Included       0.16      0.76      0.26       538

    accuracy                           0.91     24845
   macro avg       0.58      0.84      0.61     24845
weighted avg       0.98      0.91      0.94     24845



In [18]:
df_all.shape[0]

24845

## Subset Analysis

In [33]:
df_ivm.iloc[0:10]

,title,abstract,tag,decision
0,White paper on Ivermectin as a potential thera...,A group of senior doctors with vast clinical e...,Excluded,Excluded
1,What to expect from different drugs used in th...,The end of 2019 was marked by the emergence of...,Excluded,Included
2,What does the COVID-19 pandemic mean for the n...,Background: Mass drug administration (MDA) of ...,Excluded,Excluded
3,Variation in therapeutic strategies for the ma...,Aim: During the pandemic of coronavirus diseas...,Excluded,Excluded
4,Use of Ivermectin Is Associated With Lower Mor...,Background: Ivermectin was shown to inhibit se...,Included,Excluded
5,Use of ivermectin in the treatment of Covid-19...,Objectives: In this randomized open-label tria...,Included,Included
6,Use of ivermectin as a potential chemoprophyla...,Introduction: The rate of secondary attacks of...,Excluded,Included
7,An update on therapeutic repurposing strategie...,The severe acute respiratory syndrome coronavi...,Excluded,Excluded
8,"Two antioxidant 2,5-disubstituted-1,3,4-oxadia...",Repurposing of known drugs and compounds as an...,Excluded,Excluded
9,Turning the Tide: Natural Products and Natural...,The novel and highly pathogenic severe acute r...,Excluded,Excluded


In [20]:
# correctly excluded second row from df_ivm
print(f"Title: {df_ivm['title'][2]}")
print(f"Abstract: {df_ivm['abstract'][2]}")
print(f"Human: {df_ivm['tag'][2]}")
print(f"GPT: {df_ivm['decision'][2]}")

Title: What does the COVID-19 pandemic mean for the next decade of onchocerciasis control and elimination?
Abstract: Background: Mass drug administration (MDA) of ivermectin for onchocerciasis has been disrupted by the coronavirus disease 2019 (COVID-19) pandemic. Mathematical modelling can help predict how missed/delayed MDA will affect short-term epidemiological trends and elimination prospects by 2030. Method(s): Two onchocerciasis transmission models (EPIONCHO-IBM and ONCHOSIM) are used to simulate microfilarial prevalence trends, elimination probabilities and age profiles of Onchocerca volvulus microfilarial prevalence and intensity for different treatment histories and transmission settings, assuming no interruption, a 1-y (2020) interruption or a 2-y (2020-2021) interruption. Biannual MDA or increased coverage upon MDA resumption are investigated as remedial strategies. Result(s): Programmes with shorter MDA histories and settings with high pre-intervention endemicity will be th

In [21]:
# correctly included 5th row of df_ivm
print(f"Title: {df_ivm['title'][5]}")
print(f"Abstract: {df_ivm['abstract'][5]}")
print(f"Human: {df_ivm['tag'][5]}")
print(f"GPT: {df_ivm['decision'][5]}")

Title: Use of ivermectin in the treatment of Covid-19: A pilot trial
Abstract: Objectives: In this randomized open-label trial pilot study we assessed the antiviral effects and safety of various doses of ivermectin in patients with mild clinical symptoms of COVID-19. Method(s): Patients were randomly assigned to receive standard of care (SOC) treatment at hospital admission; SOC plus ivermectin 100 mcg/kg; SOC plus ivermectin 200 mcg/kg; or SOC plus ivermectin 400 mcg/kg. The primary assessed endpoint was the proportion of patients who achieved two consecutive negative SARS-CoV-2 RT PCR tests within 7 days of the start of the dosing period. This study was registered at ClinicalTrials.gov (NCT04431466). Result(s): A total of 32 patients were enrolled and randomized to treatment. SOC treatment together with ivermectin did not result in any serious adverse events. All patients exhibited a reduction in SARS-CoV-2 viral load within 7 days; however, those who received ivermectin had a more c

In [34]:
# incorrectly excluded 31th row of df_ivm
print(f"Title: {df_ivm['title'][4]}")
print(f"Abstract: {df_ivm['abstract'][4]}")
print(f"Human: {df_ivm['tag'][4]}")
print(f"GPT: {df_ivm['decision'][4]}")

Title: Use of Ivermectin Is Associated With Lower Mortality in Hospitalized Patients With Coronavirus Disease 2019: The Ivermectin in COVID Nineteen Study
Abstract: Background: Ivermectin was shown to inhibit severe acute respiratory syndrome coronavirus 2 replication in vitro, which has led to off-label use, but clinical efficacy has not been described previously. Research Question: Does ivermectin benefit hospitalized coronavirus disease 2019 (COVID-19) patients? Study Design and Methods: Charts of consecutive patients hospitalized at four Broward Health hospitals in Florida with confirmed COVID-19 between March 15 and May 11, 2020, treated with or without ivermectin were reviewed. Hospital ivermectin dosing guidelines were provided, but treatment decisions were at the treating physician's discretion. The primary outcome was all-cause in-hospital mortality. Secondary outcomes included mortality in patients with severe pulmonary involvement, extubation rates for mechanically ventilate

In [23]:
# incorrectly included 1st row of df_ivm
print(f"Title: {df_ivm['title'][1]}")
print(f"Abstract: {df_ivm['abstract'][1]}")
print(f"Human: {df_ivm['tag'][1]}")
print(f"GPT: {df_ivm['decision'][1]}")

Title: What to expect from different drugs used in the treatment of COVID-19: A study on applications and in vivo and in vitro results
Abstract: The end of 2019 was marked by the emergence of a new type of coronavirus (SARS-CoV-2), which has killed more than 240,000 people around the world so far. Several clinical studies are being performed to test possible drugs in response to the COVID-19 outbreak; however, there is still no treatment that is completely effective. Our goal in this paper is to bring together the results of main studies carried out with different drugs in order to help spread the knowledge about possible treatments for COVID-19 that have been suggested so far. Copyright © 2020 Elsevier B.V.
Human: Excluded
GPT: Included


## Reasoning Analysis

### Explanation of Decision

In [28]:
def get_gpt_call(prompt):
    memory = [{"role": "user", "content": prompt}]
    response = openai.ChatCompletion.create(
            model='gpt-3.5-turbo',
            messages=memory,
            temperature=0,
            top_p=0.5,
        )
    return response['choices'][0]['message']['content']

In [44]:
def create_prompt(df, row_num):
    prompt = f"""Explain your reasoning for the decision given with the information below.

Instructions:
You are a researcher rigorously screening titles and abstracts of scientific papers for inclusion or exclusion in a review paper. Use the criteria below to inform your decision. If any exclusion criteria are met or not all inclusion criteria are met, exclude the article. If all inclusion criteria are met, include the article. Only type "included" or "excluded" to indicate your decision. Do not type anything else.

Title: {df['title'][row_num]}

Abstract:
{df['abstract'][row_num]}

Inclusion criteria:
- Randomized controlled trials and non-randomized comparative cohort studies
- Compared ivermectin against standard of care or control group/placebo
- Included adult (age >=18) COVID-19 inpatients and/or outpatients
- Reported any of the following outcomes: time to viral clearance, duration of hospitalization, mortality incidence, incidence of progression to mechanical ventilation, all-cause adverse events, incidence of investigator-defined serious adverse events

Exclusion criteria:
- Studies that used ivermectin for prophylaxis of COVID-19
- Non-peer-reviewed articles

Decision: {df['decision'][row_num]}
"""
    return prompt

In [38]:
# correctly excluded second row from df_ivm
print(get_gpt_call(create_prompt(df_ivm, 2)))

Reasoning: The article does not meet any of the inclusion criteria or exclusion criteria listed. It is not a randomized controlled trial or comparative cohort study, it does not compare ivermectin against standard of care or control group/placebo, it does not include adult COVID-19 inpatients and/or outpatients, and it does not report any of the specified outcomes. Additionally, there is no indication that the article is non-peer-reviewed. Therefore, the article is excluded.


In [39]:
# correctly included 5th row of df_ivm
print(get_gpt_call(create_prompt(df_ivm, 5)))

Reasoning: The study meets all of the inclusion criteria. It is a randomized controlled trial that compares ivermectin against standard of care treatment in adult COVID-19 patients. The study reports outcomes related to time to viral clearance and adverse events. Additionally, the study is published in a peer-reviewed journal. Therefore, the article should be included in the review paper.


In [40]:
# incorrectly excluded 4th row of df_ivm
print(get_gpt_call(create_prompt(df_ivm, 4)))

Reasoning: The study is not a randomized controlled trial or a non-randomized comparative cohort study, which is an inclusion criteria. Therefore, the article does not meet all inclusion criteria and must be excluded.


In [41]:
# incorrectly included 1st row of df_ivm
print(get_gpt_call(create_prompt(df_ivm, 1)))

The article meets all of the inclusion criteria and none of the exclusion criteria, therefore it should be included in the review paper.


### Reflection of Incorrect Answers

In [43]:
def reflect(df, row_num):
    prompt = f"""Explain your reasoning for why the decision given was incorrect with the information below.

Instructions:
You are a researcher rigorously screening titles and abstracts of scientific papers for inclusion or exclusion in a review paper. Use the criteria below to inform your decision. If any exclusion criteria are met or not all inclusion criteria are met, exclude the article. If all inclusion criteria are met, include the article. Only type "included" or "excluded" to indicate your decision. Do not type anything else.

Title: {df['title'][row_num]}

Abstract:
{df['abstract'][row_num]}

Inclusion criteria:
- Randomized controlled trials and non-randomized comparative cohort studies
- Compared ivermectin against standard of care or control group/placebo
- Included adult (age >=18) COVID-19 inpatients and/or outpatients
- Reported any of the following outcomes: time to viral clearance, duration of hospitalization, mortality incidence, incidence of progression to mechanical ventilation, all-cause adverse events, incidence of investigator-defined serious adverse events

Exclusion criteria:
- Studies that used ivermectin for prophylaxis of COVID-19
- Non-peer-reviewed articles

Decision: {df['decision'][row_num]}
"""
    return prompt

In [45]:
# incorrectly excluded 4th row of df_ivm
print(get_gpt_call(reflect(df_ivm, 4)))

The given decision of excluding the article is incorrect. The article meets all the inclusion criteria as it is a non-randomized comparative cohort study that compares ivermectin against standard of care or control group/placebo, includes adult COVID-19 inpatients, and reports mortality incidence. The article does not use ivermectin for prophylaxis of COVID-19 and is a peer-reviewed article. Therefore, the correct decision should be "Included".


In [46]:
# incorrectly included 1st row of df_ivm
print(get_gpt_call(reflect(df_ivm, 1)))

As per the given inclusion and exclusion criteria, the decision given is incorrect. The abstract does not mention the use of ivermectin in any of the studies. Therefore, the article cannot be included based on the inclusion criteria.
